In [204]:
import torch
import numpy as np
import random
import os

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if using multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(42)


In [ ]:
from modules.unet3d import UNet3D as UNet3D_Baseline # Ensure `unet3d.py` is in `modules/`
from modules.unet3d_new import UNet3D as UNet3D_Enhanced  # Ensure `unet3d.py` is in `modules/`
import torch

import sys
import os
sys.path.append(os.path.abspath("modules"))


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_file = '14' # Test File (change as needed)

# 


# Load both models
baseline_model = UNet3D_Baseline().to(device)
enhanced_model = UNet3D_Enhanced().to(device)

baseline_model.load_state_dict(torch.load(f"models/brain_shift_model_{test_file}.pt"))
enhanced_model.load_state_dict(torch.load(f"models/brain_shift_model_{test_file}_Star.pt"))

baseline_model.eval()
enhanced_model.eval()


UNet3D(
  (encoder1): Sequential(
    (0): Conv3d(2, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (1): BatchNorm3d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.2)
    (3): Conv3d(32, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (4): BatchNorm3d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): LeakyReLU(negative_slope=0.2)
    (6): Dropout3d(p=0.2, inplace=False)
  )
  (encoder2): Sequential(
    (0): Conv3d(32, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.2)
    (3): Conv3d(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (4): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): LeakyReLU(negative_slope=0.2)
    (6): Dropout3d(p=0.2, inplace=False)
  )
  (

In [ ]:
from modules.dataset import UltrasoundDataset
from torch.utils.data import DataLoader

# Load dataset for ensemble
dataset = UltrasoundDataset(f"Test")
loader = DataLoader(dataset, batch_size=1, shuffle=False)


for pre, post, pre_landmarks, post_landmarks in loader:
    pre, post = pre.to(device), post.to(device)
    break


In [207]:
print("Number of samples:", len(dataset))


Number of samples: 1


In [208]:
with torch.no_grad():
    flow_baseline = baseline_model(pre, post)
    flow_enhanced = enhanced_model(pre, post)

ensemble_flow = 0.7 * flow_baseline + 0.3 * flow_enhanced


🔍 Shift Map Output Shape: torch.Size([1, 3, 128, 128, 128])


In [209]:
flow = ensemble_flow


In [210]:
from modules.test import compute_brain_shift

mean_shift, max_shift, shift_map = compute_brain_shift(flow)

print(f"Mean Brain Shift: {mean_shift:.3f} mm")
print(f"Max Brain Shift: {max_shift:.3f} mm")


Mean Brain Shift: 3.090 mm
Max Brain Shift: 4.094 mm


In [ ]:
import nibabel as nib
import numpy as np

# Load MNC image to get origin and shape
img = nib.load(f"Test/{test_file}/pre.mnc")
origin = img.affine[:3, 3]
original_img_shape = img.shape
spacing = np.array([0.3, 0.3, 0.3])  # voxel spacing

# Get scale factor used during resizing to (128,128,128)
scale_factor = np.array(original_img_shape) / np.array([128, 128, 128])

# Recompute voxel positions of landmarks
from modules.ground_truth import parse_tag_file
pre_landmarks, post_landmarks = parse_tag_file(f"Test/{test_file}/landmarks.tag")

pre_landmarks_voxel = (pre_landmarks - origin) / spacing
pre_landmarks_voxel = pre_landmarks_voxel / scale_factor
pre_landmarks_voxel = np.round(pre_landmarks_voxel).astype(int)

valid_landmarks = []
for x, y, z in pre_landmarks_voxel.reshape(-1, 3):
    if (0 <= x < 128) and (0 <= y < 128) and (0 <= z < 128):  
        valid_landmarks.append((x, y, z))

valid_landmarks = np.array(valid_landmarks)


In [212]:
predicted_landmark_shifts = []

for x, y, z in valid_landmarks:  
    predicted_landmark_shifts.append(shift_map[0, :, x, y, z])  

predicted_landmark_shifts = np.array(predicted_landmark_shifts)


In [213]:
valid_mask = []
valid_landmarks = []

for i, (x, y, z) in enumerate(pre_landmarks_voxel.reshape(-1, 3)):
    if (0 <= x < 128) and (0 <= y < 128) and (0 <= z < 128):  
        valid_landmarks.append((x, y, z))
        valid_mask.append(i)

valid_landmarks = np.array(valid_landmarks)


In [214]:
pre_landmarks  

array([[ 146.88212585,  142.36561584, -192.53919983],
       [ 147.32992554,  128.41355896, -194.1028595 ],
       [ 149.47290039,  100.29776001, -187.42414856],
       [ 149.08781433,   99.25779724, -158.43441772],
       [ 149.08822632,  101.0606308 , -140.65318298],
       [ 121.71213531,  141.32922363, -161.55865479],
       [ 118.0539856 ,  144.77616882, -164.82556152],
       [ 130.94386292,  139.02714539, -163.70246887],
       [ 150.02520752,  133.46078491, -180.13378906]])

In [ ]:
from modules.ground_truth import parse_tag_file


ground_truth_vectors = (post_landmarks - pre_landmarks)
ground_truth_vectors = ground_truth_vectors[valid_mask]


In [ ]:
import numpy as np
from scipy.stats import iqr

# Convert 3D vectors to shift magnitudes
predicted_magnitudes = np.linalg.norm(predicted_landmark_shifts, axis=1)  # (9,)
ground_truth_magnitudes = np.linalg.norm(ground_truth_vectors, axis=1)  # (9,)

# normalize vectors to a unit vector
gt_norm = ground_truth_vectors / (np.linalg.norm(ground_truth_vectors, axis=1, keepdims=True) + 1e-6)
pred_norm = predicted_landmark_shifts / (np.linalg.norm(predicted_landmark_shifts, axis=1, keepdims=True) + 1e-6)

print(f'for {test_file} landmarks:')

# Mean Absolute Error (MAE)
mae = np.mean(np.abs(predicted_magnitudes - ground_truth_magnitudes))
mae_sd = np.std(np.abs(predicted_magnitudes - ground_truth_magnitudes))
print(f"Mean Absolute Error (MAE): {mae:.3f} ± {mae_sd:.3f} mm")

# Median Absolute Error
median_ae = np.median(np.abs(predicted_magnitudes - ground_truth_magnitudes))
mae_iqr = iqr(np.abs(predicted_magnitudes - ground_truth_magnitudes))
print(f"Median Absolute Error (MedAE): {median_ae:.3f} mm")
print(f"MAE IQR: {mae_iqr:.3f} mm")

# Root Mean Squared Error (RMSE)
rmse = np.sqrt(np.mean((predicted_magnitudes - ground_truth_magnitudes) ** 2))
print(f"Root Mean Squared Error (RMSE): {rmse:.3f} mm")

# Cosine Similarity
cosine_sim = np.sum(gt_norm * pred_norm, axis=1)  # Dot product
cosine_sim = np.clip(cosine_sim, -1.0, 1.0)
angular_error = np.arccos(cosine_sim) * (180 / np.pi)
mean_cosine_sim = np.mean(cosine_sim)
print(f"Mean Cosine Similarity: {mean_cosine_sim:.3f}")

# Mean Angular Error
mean_angular_error = np.mean(angular_error)
angular_sd = np.std(angular_error)
print(f"Mean Angular Error: {mean_angular_error:.2f}° ± {angular_sd:.2f}°")

# Median Angular Error
median_angular_error = np.median(angular_error)
angular_iqr = iqr(angular_error)

print(f"Median Angular Error: {median_angular_error:.2f}°")
print(f"Angular Error IQR: {angular_iqr:.2f}°")


for 14 landmarks:
Mean Absolute Error (MAE): 0.983 ± 0.647 mm
Median Absolute Error (MedAE): 0.716 mm
MAE IQR: 0.618 mm
Root Mean Squared Error (RMSE): 1.176 mm
Mean Cosine Similarity: 0.457
Mean Angular Error: 58.72° ± 33.87°
Median Angular Error: 42.49°
Angular Error IQR: 46.43°
